# 🦠 Global Hantavirus Surveillance (2026) — Complete EDA & Risk Analysis

**2,000 cases · 10 countries · 5 virus strains · Environmental & demographic intelligence**

> *"The best way to fight a disease is to understand it."* — WHO

### What This Notebook Covers
1. Dataset Overview & Data Quality
2. Global Case Distribution
3. Virus Strain Analysis
4. Transmission Patterns
5. Exposure Source Intelligence
6. Patient Demographics
7. Clinical Outcomes — Hospitalization & Fatality
8. Environmental Correlations
9. Recovery Time Analysis
10. Time Series — Outbreak Progression
11. Symptom Cluster Analysis
12. Fatality Risk Classifier (ML Model)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_predict
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 110

# Color palette
RED    = '#E63946'
ORANGE = '#F4A261'
GOLD   = '#E9C46A'
TEAL   = '#2A9D8F'
BLUE   = '#264653'
GREEN  = '#52B788'
PURPLE = '#7B2D8B'
SMOKE  = '#8D99AE'

STRAIN_COLORS = {
    'Sin Nombre': RED,
    'Seoul':      ORANGE,
    'Andes':      TEAL,
    'Dobrava':    PURPLE,
    'Puumala':    GREEN,
}
COUNTRY_COLORS = {
    'Bolivia': '#E63946', 'Argentina': '#F4A261', 'Canada': '#2A9D8F',
    'USA': '#264653',     'Brazil': '#52B788',    'Uruguay': '#7B2D8B',
    'Mexico': '#E9C46A',  'Paraguay': '#A8DADC',  'Chile': '#457B9D', 'Peru': '#1D3557'
}

print("✅ Libraries loaded — Surveillance active")

## 1. Dataset Overview & Data Quality

In [ ]:
df = pd.read_csv("/kaggle/input/global-hantavirus-surveillance-dataset-2026/global_hantavirus_surveillance_dataset_2026.csv")

# Parse dates
df['report_date'] = pd.to_datetime(df['report_date'], dayfirst=True)
df['month']  = df['report_date'].dt.month
df['year']   = df['report_date'].dt.year
df['month_name'] = df['report_date'].dt.strftime('%b')

# Binary targets
df['fatal']         = (df['fatality'] == 'Yes').astype(int)
df['hospitalized']  = (df['hospitalization'] == 'Yes').astype(int)
df['rodent_route']  = (df['transmission_type'] == 'Rodent-to-Human').astype(int)

print(f"Shape:          {df.shape}")
print(f"Date range:     {df['report_date'].min().date()} → {df['report_date'].max().date()}")
print(f"Countries:      {df['country'].nunique()}")
print(f"Virus strains:  {df['virus_strain'].nunique()}")
print(f"Case fatality:  {df['fatal'].mean()*100:.1f}%")
print(f"Hospitalized:   {df['hospitalized'].mean()*100:.1f}%")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum()>0].to_string())
df.head(3)

In [ ]:
df.describe().round(2)

## 2. Global Case Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

country_counts = df['country'].value_counts().sort_values()
colors_c = [COUNTRY_COLORS.get(c, SMOKE) for c in country_counts.index]
country_counts.plot.barh(ax=axes[0], color=colors_c, edgecolor='white', linewidth=0.5)
axes[0].set_title('Hantavirus Cases by Country', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Cases')
for bar, val in zip(axes[0].patches, country_counts.values):
    axes[0].text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
                 str(val), va='center', fontsize=9)

# Fatality rate by country
cfr_country = df.groupby('country')['fatal'].mean()*100
cfr_sorted = cfr_country.sort_values(ascending=True)
bar_colors = [RED if v > 10 else ORANGE if v > 7 else TEAL for v in cfr_sorted.values]
cfr_sorted.plot.barh(ax=axes[1], color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Case Fatality Rate by Country (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Case Fatality Rate (%)')
axes[1].axvline(df['fatal'].mean()*100, color='black', linestyle='--', alpha=0.5,
                label=f'Overall CFR: {df["fatal"].mean()*100:.1f}%')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Virus Strain Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

strain_counts = df['virus_strain'].value_counts()
wedges, texts, autotexts = axes[0].pie(
    strain_counts, labels=strain_counts.index, autopct='%1.1f%%',
    colors=[STRAIN_COLORS.get(s, SMOKE) for s in strain_counts.index],
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    startangle=90)
for at in autotexts: at.set_fontsize(9)
axes[0].set_title('Distribution by Virus Strain', fontsize=13, fontweight='bold')

# CFR by strain
cfr_strain = df.groupby('virus_strain')['fatal'].mean()*100
cfr_strain.sort_values().plot.barh(
    ax=axes[1],
    color=[STRAIN_COLORS.get(s, SMOKE) for s in cfr_strain.sort_values().index],
    edgecolor='white', linewidth=0.5)
axes[1].axvline(df['fatal'].mean()*100, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Case Fatality Rate by Strain (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('CFR (%)')

# Hospitalization rate by strain
hosp_strain = df.groupby('virus_strain')['hospitalized'].mean()*100
hosp_strain.sort_values().plot.barh(
    ax=axes[2],
    color=[STRAIN_COLORS.get(s, SMOKE) for s in hosp_strain.sort_values().index],
    edgecolor='white', linewidth=0.5)
axes[2].set_title('Hospitalization Rate by Strain (%)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Hospitalization Rate (%)')

plt.tight_layout()
plt.show()

print("Strain statistics:")
print(df.groupby('virus_strain')[['fatal','hospitalized','recovery_days']].agg(
    {'fatal':'mean','hospitalized':'mean','recovery_days':'mean'}).mul({'fatal':100,'hospitalized':100,'recovery_days':1}).round(2).to_string())

## 4. Transmission Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

trans_counts = df['transmission_type'].value_counts()
axes[0].pie(trans_counts, labels=trans_counts.index, autopct='%1.1f%%',
            colors=[RED, TEAL], wedgeprops={'edgecolor': 'white', 'linewidth': 2},
            startangle=90)
axes[0].set_title('Transmission Route Distribution', fontsize=13, fontweight='bold')

# CFR & hospitalization by transmission type
trans_stats = df.groupby('transmission_type')[['fatal', 'hospitalized']].mean() * 100
trans_stats.plot.bar(ax=axes[1], color=[RED, TEAL], edgecolor='white', linewidth=0.5, width=0.5)
axes[1].set_title('Outcomes by Transmission Type (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(['Fatality Rate', 'Hospitalization Rate'], fontsize=9)

plt.tight_layout()
plt.show()

print("Transmission type × outcome:")
print(df.groupby('transmission_type')[['fatal','hospitalized','recovery_days','quarantine_days']].mean().round(3).to_string())

## 5. Exposure Source Intelligence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

exp_counts = df['exposure_source'].value_counts().sort_values()
colors_exp = [RED, ORANGE, GOLD, TEAL, GREEN, BLUE]
exp_counts.plot.barh(ax=axes[0], color=colors_exp, edgecolor='white', linewidth=0.5)
axes[0].set_title('Cases by Exposure Source', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Cases')

cfr_exp = df.groupby('exposure_source')['fatal'].mean()*100
cfr_exp.sort_values().plot.barh(
    ax=axes[1],
    color=[RED if v > 10 else ORANGE if v > 7 else TEAL for v in cfr_exp.sort_values().values],
    edgecolor='white', linewidth=0.5)
axes[1].axvline(df['fatal'].mean()*100, color='black', linestyle='--', alpha=0.5,
                label='Overall CFR')
axes[1].set_title('Case Fatality Rate by Exposure Source (%)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6. Patient Demographics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Age distribution
df['patient_age'].plot.hist(bins=30, ax=axes[0,0], color=TEAL, edgecolor='white', alpha=0.85)
axes[0,0].axvline(df['patient_age'].mean(), color=RED, linewidth=2, linestyle='--',
                  label=f'Mean: {df["patient_age"].mean():.1f} yrs')
axes[0,0].set_title('Patient Age Distribution', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Age (years)')
axes[0,0].legend()

# Age vs fatality
fatal_ages = df.groupby('fatal')['patient_age'].mean()
axes[0,1].bar(['Survived', 'Fatal'], fatal_ages.values,
               color=[TEAL, RED], edgecolor='white', width=0.4)
axes[0,1].set_title('Avg Age: Survived vs Fatal', fontsize=13, fontweight='bold')
axes[0,1].set_ylabel('Mean Age')
for bar, val in zip(axes[0,1].patches, fatal_ages.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   f'{val:.1f}', ha='center', fontsize=12, fontweight='bold')

# Gender distribution
gender_counts = df['gender'].value_counts()
axes[1,0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
              colors=[BLUE, ORANGE], wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1,0].set_title('Gender Distribution', fontsize=13, fontweight='bold')

# CFR by gender
cfr_gender = df.groupby('gender')['fatal'].mean()*100
axes[1,1].bar(cfr_gender.index, cfr_gender.values,
               color=[BLUE, ORANGE], edgecolor='white', width=0.4)
axes[1,1].set_title('Case Fatality Rate by Gender (%)', fontsize=13, fontweight='bold')
axes[1,1].set_ylabel('CFR (%)')
for bar, val in zip(axes[1,1].patches, cfr_gender.values):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                   f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Age groups
df['age_group'] = pd.cut(df['patient_age'], bins=[0,17,30,45,60,80],
                          labels=['<18','18-30','31-45','46-60','60+'])
print("CFR by age group:")
print(df.groupby('age_group')['fatal'].mean().mul(100).round(2).to_string())

## 7. Clinical Outcomes — Hospitalization & Fatality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Hospitalization × fatality cross-tab
hosp_fatal = pd.crosstab(df['hospitalization'], df['fatality'], normalize='index')*100
hosp_fatal.plot.bar(ax=axes[0], color=[TEAL, RED], edgecolor='white', linewidth=0.5, width=0.5)
axes[0].set_title('Fatality Rate by Hospitalization Status (%)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['Survived', 'Fatal'], fontsize=9)

# Age group × fatality heatmap
age_strain_cfr = df.groupby(['age_group', 'virus_strain'])['fatal'].mean()*100
age_strain_pivot = age_strain_cfr.unstack(fill_value=0)
sns.heatmap(age_strain_pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'CFR (%)'})
axes[1].set_title('Case Fatality Rate: Age Group × Virus Strain (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Cases hospitalized AND fatal: {len(df[(df['hospitalized']==1)&(df['fatal']==1)]):,}")
print(f"Fatal without hospitalization: {len(df[(df['hospitalized']==0)&(df['fatal']==1)]):,}")

## 8. Environmental Correlations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

env_vars = [
    ('temperature_celsius', 'Temperature (°C)', ORANGE),
    ('humidity_percent', 'Humidity (%)', BLUE),
    ('rodent_presence_index', 'Rodent Presence Index', RED),
    ('population_density', 'Population Density', PURPLE),
    ('air_quality_index', 'Air Quality Index', GREEN),
]

for (col, label, color), ax in zip(env_vars, axes.flatten()):
    fatal_data = df[df['fatal']==1][col]
    surv_data  = df[df['fatal']==0][col]
    ax.hist(surv_data, bins=25, color=TEAL, alpha=0.6, density=True, label='Survived', edgecolor='white')
    ax.hist(fatal_data, bins=25, color=RED, alpha=0.7, density=True, label='Fatal', edgecolor='white')
    corr = df[col].corr(df['fatal'])
    ax.set_title(f'{label}\nr = {corr:.3f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

# Correlation matrix of all numeric variables
corr_cols = ['patient_age', 'temperature_celsius', 'humidity_percent',
             'rodent_presence_index', 'quarantine_days', 'population_density',
             'air_quality_index', 'fatal', 'hospitalized']
corr_matrix = df[corr_cols].corr()
corr_matrix.index = ['Age', 'Temp', 'Humidity', 'Rodent Idx', 'Quarantine',
                     'Pop Density', 'AQI', 'Fatal', 'Hospitalized']
corr_matrix.columns = corr_matrix.index
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=axes[1,2], square=True, linewidths=0.5,
            vmin=-0.5, vmax=0.5, annot_kws={'size': 8})
axes[1,2].set_title('Correlation Matrix', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Recovery Time Analysis

In [ ]:
df_rec = df[df['recovery_days'].notna()].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

df_rec['recovery_days'].plot.hist(bins=30, ax=axes[0], color=GREEN, edgecolor='white', alpha=0.85)
axes[0].axvline(df_rec['recovery_days'].mean(), color=RED, linewidth=2, linestyle='--',
                label=f'Mean: {df_rec["recovery_days"].mean():.1f} days')
axes[0].set_title('Recovery Days Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Recovery Days')
axes[0].legend()

# Recovery by virus strain
sns.boxplot(data=df_rec, x='virus_strain', y='recovery_days', ax=axes[1],
            palette=STRAIN_COLORS, linewidth=1.0)
axes[1].set_title('Recovery Days by Virus Strain', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

# Recovery vs age scatter
axes[2].scatter(df_rec['patient_age'], df_rec['recovery_days'],
                alpha=0.2, s=15, c=df_rec['fatal'], cmap='RdYlGn_r', edgecolors='none')
corr_rec = df_rec['patient_age'].corr(df_rec['recovery_days'])
axes[2].set_title(f'Age vs Recovery Days (r={corr_rec:.3f})', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Patient Age'); axes[2].set_ylabel('Recovery Days')

plt.tight_layout()
plt.show()

print("Avg recovery days by strain:")
print(df_rec.groupby('virus_strain')['recovery_days'].mean().round(1).to_string())
print(f"\nMissing recovery_days: {df['recovery_days'].isna().sum()} (fatal cases)")

## 10. Time Series — Outbreak Progression

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Monthly cases
monthly = df.groupby(df['report_date'].dt.to_period('M')).size()
monthly.index = monthly.index.to_timestamp()
axes[0,0].plot(monthly.index, monthly.values, color=RED, linewidth=2.2, marker='o', markersize=4)
axes[0,0].fill_between(monthly.index, monthly.values, alpha=0.12, color=RED)
axes[0,0].set_title('Monthly Case Count', fontsize=13, fontweight='bold')
axes[0,0].set_ylabel('Cases')

# Monthly fatality rate
monthly_cfr = df.groupby(df['report_date'].dt.to_period('M'))['fatal'].mean()*100
monthly_cfr.index = monthly_cfr.index.to_timestamp()
axes[0,1].plot(monthly_cfr.index, monthly_cfr.values, color=ORANGE, linewidth=2, marker='s', markersize=4)
axes[0,1].set_title('Monthly Case Fatality Rate (%)', fontsize=13, fontweight='bold')
axes[0,1].set_ylabel('CFR (%)')

# Cases by month (seasonal)
month_cases = df.groupby('month').size()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1,0].bar(range(1,13), month_cases.values, color=TEAL, edgecolor='white', linewidth=0.5, alpha=0.85)
axes[1,0].set_xticks(range(1,13))
axes[1,0].set_xticklabels(month_names)
axes[1,0].set_title('Seasonal Case Distribution', fontsize=13, fontweight='bold')
axes[1,0].set_ylabel('Cases')

# Strain trend over months
for strain, color in STRAIN_COLORS.items():
    strain_monthly = df[df['virus_strain']==strain].groupby(
        df['report_date'].dt.to_period('M')).size()
    strain_monthly.index = strain_monthly.index.to_timestamp()
    axes[1,1].plot(strain_monthly.index, strain_monthly.values,
                   color=color, linewidth=1.8, label=strain, alpha=0.85)
axes[1,1].set_title('Monthly Cases by Virus Strain', fontsize=13, fontweight='bold')
axes[1,1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

## 11. Symptom Cluster Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

symptom_counts = df['symptoms'].value_counts()
colors_sym = [RED, ORANGE, GOLD, TEAL, GREEN]
symptom_counts.sort_values().plot.barh(ax=axes[0], color=colors_sym, edgecolor='white', linewidth=0.5)
axes[0].set_title('Symptom Presentation Frequency', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cases')

# Fatality rate by symptom
cfr_sym = df.groupby('symptoms')['fatal'].mean()*100
cfr_sym.sort_values().plot.barh(ax=axes[1],
    color=[RED if v > 10 else ORANGE if v > 7 else TEAL for v in cfr_sym.sort_values().values],
    edgecolor='white', linewidth=0.5)
axes[1].axvline(df['fatal'].mean()*100, color='black', linestyle='--', alpha=0.5,
                label='Overall CFR')
axes[1].set_title('Case Fatality Rate by Symptom Cluster (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('CFR (%)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Rodent presence index vs symptom severity proxy
print("Avg rodent presence index by symptom cluster:")
print(df.groupby('symptoms')['rodent_presence_index'].mean().sort_values(ascending=False).round(2).to_string())

## 12. 🤖 Fatality Risk Classifier

In [ ]:
for col in ['country', 'virus_strain', 'transmission_type', 'exposure_source',
            'gender', 'symptoms', 'hospitalization']:
    df[col+'_enc'] = LabelEncoder().fit_transform(df[col].astype(str))

df['age_group_enc'] = LabelEncoder().fit_transform(df['age_group'].astype(str))

feats = ['patient_age', 'age_group_enc', 'country_enc', 'virus_strain_enc',
         'transmission_type_enc', 'exposure_source_enc', 'gender_enc',
         'symptoms_enc', 'hospitalization_enc', 'temperature_celsius',
         'humidity_percent', 'rodent_presence_index', 'quarantine_days',
         'population_density', 'air_quality_index']

X = df[feats].values
y = df['fatal'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in [
    ('Random Forest',     RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42)),
]:
    roc = cross_val_score(clf, X, y, cv=skf, scoring='roc_auc')
    f1  = cross_val_score(clf, X, y, cv=skf, scoring='f1')
    print(f"{name:25s}  ROC-AUC={roc.mean():.4f}±{roc.std():.4f}  F1={f1.mean():.4f}")

In [ ]:
gb = GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42)
gb.fit(X, y)

fi = pd.Series(gb.feature_importances_, index=feats).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

fi.plot.barh(ax=axes[0],
    color=[RED if v > 0.07 else TEAL for v in fi.values],
    edgecolor='white', linewidth=0.4)
axes[0].set_title('Feature Importance — Fatality Predictor', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Relative Importance')

y_pred = cross_val_predict(gb, X, y, cv=3)
ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=axes[1],
    colorbar=False, cmap='Blues',
    display_labels=['Survived', 'Fatal'])
axes[1].set_title('Confusion Matrix (3-Fold CV)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n", classification_report(y, y_pred, target_names=['Survived','Fatal']))

## 📋 Key Findings

### 🦠 Epidemiology
- Overall **Case Fatality Rate: ~7.75%** — consistent with documented hantavirus HPS/HFRS rates
- **Rodent-to-Human** and **Human-to-Human** transmission are near-equal — unusual vs real-world epidemiology
- **Hospitalization rate: ~50%** — reflects the severe systemic nature of hantavirus infections

### 🌍 Geography
- Cases distributed across 10 countries, predominantly South American (consistent with Andes/Sin Nombre strains)
- CFR varies significantly by country — reflects healthcare infrastructure differences

### 🧬 Virus Strains
- **Sin Nombre** (USA/Canada): associated with highest pulmonary severity
- **Puumala** (typically Europe): generally milder course in real outbreaks
- **Andes**: the only hantavirus with confirmed human-to-human transmission

### 📊 Risk Factors (ML Model)
- **Hospitalization status** is the strongest predictor — severity marker
- **Age** is a significant risk factor — older patients have higher CFR
- **Rodent presence index** contributes — environmental exposure risk
- Environmental variables (temperature, humidity, AQI) add secondary signal

---
*If this notebook was useful, please upvote! 🙏*